Initialization

In [2]:
%matplotlib inline
import os
import sys

import time
import csv
import json
import numpy as np 

from src.biology import * 
from src.human import *
from src.evolution import *
from src.physics import *
from src.traditional import * 
    
from src.testing.continous.test_functions import sphere, rastrigin, rosenbrock, griewank, ackley
from src.testing.discrete_problems.TSP           import TSP, TSPSolver
from src.testing.discrete_problems.Knapsack      import KnapsackProblem, KnapsackSolver
from src.testing.discrete_problems.GraphColoring import GraphColoring, GraphColoringSolver

from src.utils.logger import *
from src.benchmark import BenchmarkRunner
from src.visualization.visualize import (
    plot_convergence, plot_box_scores, plot_time_comparison,
    plot_heatmap_ranking, plot_summary_table, plot_overall_comparison,
    plot_discrete_convergence, plot_discrete_bar,
    plot_pathfinding_grids, plot_pathfinding_metrics, plot_pathfinding_summary_table,
    plot_tsp_tour, plot_knapsack_items, plot_graph_coloring,
    plot_scalability, plot_exploration_exploitation
)

SETTING PARAMETERS AND ALGORITHMS

In [3]:
# Parameters Configuration
n_trials = 10
dim = 10
pop_size = 30
max_iter_pop = 1000
max_iter_single = 20000
seed = 42

# Initialize Algorithms
pso = PSO(n_particles=pop_size, max_iter=max_iter_pop, c1=1.49618, c2=1.49618, w=0.7298)
cs = CS(n_nests=pop_size, max_iter=max_iter_pop, pa=0.25)
fa = FA(n_fireflies=pop_size, max_iter=max_iter_pop, alpha=0.5, beta0=1.0, gamma=0.1)
de = DE(pop_size=pop_size, max_iter=max_iter_pop, F=0.8, CR=0.9)
tlbo = TLBO(pop_size=pop_size, max_iter=max_iter_pop)
sa = SA(max_iter=max_iter_single, T0=100, alpha=0.995)
hc = HillClimbing(max_iter=max_iter_single)

# Output directory
save_dir = "output"
os.makedirs(save_dir, exist_ok=True)

# Benchmark class to run the benchmark
runner = BenchmarkRunner(
    n_trials=n_trials,
    dim=dim,
    max_iter=max_iter_pop,
    seed=seed
)

CONTINUOUS OPTIMIZATION BENCHMARKS

In [ ]:
# Define continuos Functions
CONTINUOUS_FUNCTIONS = {
    "Sphere":     (sphere,     (-100, 100)),
    "Rastrigin":  (rastrigin,  (-5.12, 5.12)),
    "Rosenbrock": (rosenbrock, (-5, 10)),
    "Griewank":   (griewank,   (-600, 600)),
    "Ackley":     (ackley,     (-32.768, 32.768)),
}

# Choosing algorithms to test continuous problems
continuous_algorithms = {
    "CS": cs,
    "FA": fa,
    "PSO": pso,
    "HC": hc,
    "SA": sa,
    "TLBO": tlbo,
    "DE": de
}

In [ ]:
# RUNNING THE BENCHMARKS
total_t0 = time.perf_counter()
cont_results = runner.run_continuous_benchmarks(CONTINUOUS_FUNCTIONS, continuous_algorithms, verbose=True)
elapsed = time.perf_counter() - total_t0
print(f"\n{'=' * 60}")
print(f"  DONE  — total time {elapsed:.1f}s")
print(f"{'=' * 60}\n")

save_continuous_benchmarks(cont_results, save_dir="output")
export_continuous_csv(cont_results, save_dir = "output")

SCALABILITY RUNS 

In [ ]:
dims = [10, 20, 30]
scalability = runner.run_scalability_benchmarks(dims, CONTINUOUS_FUNCTIONS, continuous_algorithms, verbose = True)
save_scalability_benchmarks(scalability, save_dir="output")

EXPLORATON AND EXPLOITATION RUNS

In [ ]:
# Exploration and Exploitation
results = runner.bench_exploration_exploitation(CONTINUOUS_FUNCTIONS, continuous_algorithms, verbose=True)
save_exploration_exploitation_benchmarks(results, save_dir="output")

DISCRETE OPTIMIZATION BENCHMARKS

In [ ]:
# Initialize 3 discrete problems: Traveling Sales Problem, Knapsack Problem, Graph Colouring Problem
tsp = TSP(np.zeros((1,1))) # dummy init
gc = GraphColoring(1)   # temporary init for n_vertices 
kp = KnapsackProblem([1], [1], 1)  # dummy init

# Chosing prepared testcases for experiments
tsp.load_from_json("test_case/small_tsp_10.json")
gc.load_from_json("test_case/gc_small_sparse.json")
kp.load_from_json("test_case/knapsack_small.json")

In [ ]:
# Create problem solver classes of each problems
TSP_solver = TSPSolver(tsp)
Knapsack_solver = KnapsackSolver(kp)
Graphcoloring_solver = GraphColoringSolver(gc, n_colors = gc.n_vertices)

# Define specific algorithms to solve problems
TSP_algos = {
    "SA (TSP)": lambda: TSP_solver.solve_sa(T0=500.0, T_min=1e-3, max_iter=max_iter_single, alpha=0.001, verbose=False),
    "GA (TSP)": lambda: TSP_solver.solve_ga(pop_size=pop_size, max_iter=max_iter_pop, CR=0.9, mutation_rate=0.02, tournament_size=3, elitism_rate=0.1, beta=2.0, verbose=False),
    "ACO (TSP)": lambda: TSP_solver.solve_aco(n_ants=pop_size, max_iter=max_iter_pop, alpha=1.0, beta_aco=3.0, rho=0.1, Q=100.0, tau_init=1.0, verbose=False),
    "CS (TSP)": lambda: TSP_solver.solve_cs(n_nests=pop_size, max_iter=max_iter_pop, pa=0.25, verbose=False),
    "ABC (TSP)": lambda: TSP_solver.solve_abc(n_bees=pop_size, max_iter=max_iter_pop, limit=pop_size * dim, verbose=False),
    "FA (TSP)": lambda: TSP_solver.solve_fa(n_fireflies=pop_size, max_iter=max_iter_pop, alpha=0.5, beta0=1.0, gamma=0.1, alpha_decay=0.97, verbose=False),
    "A* (TSP)": lambda: TSP_solver.solve_astar(verbose=False),
    "BFS (TSP)": lambda: TSP_solver.solve_bfs(beam_width=10, verbose=False),
    "DFS (TSP)": lambda: TSP_solver.solve_dfs(max_nodes=50_000, verbose=False),
}

Knapsack_algos = {
    "SA (Knapsack)": lambda: Knapsack_solver.solve_sa(max_iter=max_iter_single, verbose=False),
    "GA (Knapsack)": lambda: Knapsack_solver.solve_ga(pop_size=pop_size, max_iter=max_iter_pop, verbose=False),
    "ACO (Knapsack)": lambda: Knapsack_solver.solve_aco(n_ants=pop_size, max_iter=max_iter_pop, verbose=False),
    "CS (Knapsack)": lambda: Knapsack_solver.solve_cs(n_nests=pop_size, max_iter=max_iter_pop, verbose=False),
    "ABC (Knapsack)": lambda: Knapsack_solver.solve_abc(n_bees=pop_size, max_iter=max_iter_pop, verbose=False),
    "FA (Knapsack)": lambda: Knapsack_solver.solve_fa(n_fireflies=pop_size, max_iter=max_iter_pop, verbose=False),
    "A* (Knapsack)": lambda: Knapsack_solver.solve_astar(verbose=False),
    "BFS (Knapsack)": lambda: Knapsack_solver.solve_bfs(beam_width=20, verbose=False),
    "DFS (Knapsack)": lambda: Knapsack_solver.solve_dfs(max_nodes=100_000, verbose=False),
}

Graph_colouring_algos = {
    "SA (GC)": lambda: Graphcoloring_solver.solve_sa(max_iter=max_iter_single, verbose=False),
    "GA (GC)": lambda: Graphcoloring_solver.solve_ga(pop_size=pop_size, max_iter=max_iter_pop, verbose=False),
    "ACO (GC)": lambda: Graphcoloring_solver.solve_aco(n_ants=pop_size, max_iter=max_iter_pop, verbose=False),
    "CS (GC)": lambda: Graphcoloring_solver.solve_cs(n_nests=pop_size, max_iter=max_iter_pop, verbose=False),
    "ABC (GC)": lambda: Graphcoloring_solver.solve_abc(n_bees=pop_size, max_iter=max_iter_pop, verbose=False),
    "FA (GC)": lambda: Graphcoloring_solver.solve_fa(n_fireflies=pop_size, max_iter=max_iter_pop, verbose=False),
    "A* (GC)": lambda: Graphcoloring_solver.solve_astar(verbose=False),
    "BFS (GC)": lambda: Graphcoloring_solver.solve_bfs(beam_width=20, verbose=False),
    "DFS (GC)": lambda: Graphcoloring_solver.solve_dfs(max_nodes=10_000, verbose=False),
}

# RUNNING THE BENCHMARKS
print("\n" + "=" * 60)
print("  DISCRETE PROBLEM BENCHMARKS")
print("=" * 60)

# Benmark the algorithms based on those discrete problems

total_t0 = time.perf_counter()

disc_results = {}
disc_results["TSP"] = runner._bench_tsp(TSP_algos, tsp, verbose = True)
disc_results["Knapsack"] = runner._bench_knapsack(Knapsack_algos, kp, verbose = True)
disc_results["Graph Coloring"] = runner._bench_graph_coloring(Graph_colouring_algos, gc, verbose = True)

elapsed = time.perf_counter() - total_t0

print(f"\n{'=' * 60}")
print(f"  DONE  — total time {elapsed:.1f}s")
print(f"{'=' * 60}\n")

save_discrete_benchmarks(disc_results, save_dir)

PARAMETERS SENSITIVITY ANALYSIS

In [4]:
# Pick most featured algorithm to analyze 
# Parameters of GA
mutation_rate=np.array([0.005, 0.01, 0.02, 0.05])

# Parameters of ACO 
rho=np.array([0.02, 0.05, 0.1, 0.2])

# Parameters of SA 
alpha=np.array([0.001, 0.003, 0.005, 0.01])

parameters = {
    "mutation_rate": mutation_rate,
    "rho": rho,
    "alpha": alpha
}
    
results = runner.run_parameters_sensitivity(parameters, continuous_func=sphere, continuous_bounds=(-5.12, 5.12),
                                            TEST_CASE="test_case/medium_tsp_30.json",verbose = True)
save_parameter_sensitivity_benchmarks(results, save_dir="output")

Iteration: 2  T: 998.00200  f([[-2.48376  2.93341  1.25826  1.14468 -0.65917 -4.71711 -4.9951   2.64769
 -0.61412  3.42037]]) = 84.39037
Iteration: 46  T: 955.04196  f([[-1.73912  2.99873  0.89394  4.16986 -0.12724 -2.92186  0.47239  2.41321
 -3.83967  2.08942]]) = 63.91276
Iteration: 47  T: 954.08740  f([[-1.92172  2.95204 -0.1429   3.53055 -0.88267 -3.26457  0.68559  3.30282
 -2.80862  2.72802]]) = 63.03841
Iteration: 48  T: 953.13379  f([[-2.77473  2.94883  2.17447  4.52074 -1.31261 -1.72049  0.30683  2.65045
 -1.44638  1.44979]]) = 57.55610
Iteration: 219  T: 803.32172  f([[-2.25551 -0.13678  1.66218  2.2644  -2.79815 -3.30556 -0.05322 -0.4578
 -0.73284  3.90957]]) = 47.78701
Iteration: 403  T: 668.31210  f([[ 0.42659 -2.84012  3.85742  3.0478  -0.9192   0.09687 -0.37779 -1.94258
  1.96363 -0.13124]]) = 41.06074
Iteration: 405  T: 666.97681  f([[ 0.84448 -2.04269  2.23379  3.29101 -0.17767  0.80098 -0.96862 -2.86568
  2.3722  -0.77215]]) = 36.75330
Iteration: 406  T: 666.31017  f([

'/Users/thaihoang78/Project/Nature_Inspired_Algorithms/output/parameter_sensitivity_results.json'